# Regional & Municipality Real Estate Market Analysis

This notebook moves from integrated market data to **market segmentation and ranking**. It identifies regional and municipality-level differences in price levels, transaction activity and demographic dynamics.

**Pipeline:** integrated panel → KPI construction → regional ranking → municipality ranking → market-size segmentation → growth quadrants → interpretation.

> Analytical note: rankings are descriptive. They do not imply causality, investment advice or representative transaction prices.

## 1. Setup

The notebook reconstructs the integrated panel from repository data so it can be executed independently of previous notebook state.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

QUOTATIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
TRANSACTIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
POPULATION_DIR = PROJECT_ROOT / 'data' / 'raw' / 'population'
for path in [QUOTATIONS_DIR, TRANSACTIONS_DIR, POPULATION_DIR]:
    assert path.exists(), f'Missing source directory: {path}'

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Build the municipality-semester market panel

OMI quotations are aggregated to municipality + semester. NTN and population are annual measures joined to both semesters of the corresponding year.

In [ ]:
quotation_parts = []
for path in sorted(QUOTATIONS_DIR.glob('omi_quotations_*.csv')):
    match = re.search(r'_(\d{4})_(S[12])$', path.stem)
    if not match:
        continue
    year, semester = int(match.group(1)), match.group(2)
    df = pd.read_csv(path, sep=';', low_memory=False)
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    required = ['Comune_ISTAT', 'Descr_Tipologia', 'Compr_min', 'Compr_max', 'Regione']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'{path.name}: missing columns {missing}')
    df = df[required].copy()
    df['year'], df['semester'] = year, semester
    quotation_parts.append(df)

omi = pd.concat(quotation_parts, ignore_index=True)
omi['municipality_code'] = omi['Comune_ISTAT'].astype('string').str.strip()
omi['Compr_min'] = pd.to_numeric(omi['Compr_min'], errors='coerce')
omi['Compr_max'] = pd.to_numeric(omi['Compr_max'], errors='coerce')
omi['price_m2'] = omi[['Compr_min', 'Compr_max']].mean(axis=1)
residential = omi[omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)].copy()

price_panel = (residential.dropna(subset=['municipality_code', 'price_m2'])
    .groupby(['year', 'semester', 'municipality_code'], as_index=False)
    .agg(price_m2=('price_m2', 'median'), quotation_obs=('price_m2', 'size'), region=('Regione', 'first')))

transaction_parts = []
for folder in sorted(p for p in TRANSACTIONS_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year = int(folder.name)
    files = [p for p in folder.iterdir() if p.is_file()]
    lista_path = next(p for p in files if 'lista-com' in p.name.lower())
    res_path = next(p for p in files if 'valori-res' in p.name.lower())
    lista = pd.read_csv(lista_path, sep=';', decimal=',')
    res = pd.read_csv(res_path, sep=';', decimal=',')
    lista.columns = [str(c).strip() for c in lista.columns]
    res.columns = [str(c).strip() for c in res.columns]
    lista_code = next(c for c in lista.columns if re.search(r'codcom$', c, re.I))
    res_code = next(c for c in res.columns if re.search(r'codcom$', c, re.I))
    ntn_candidates = [c for c in res.columns if re.fullmatch(r'NTN_?'+str(year), c, re.I)]
    if not ntn_candidates:
        ntn_candidates = [c for c in res.columns if re.match(r'NTN', c, re.I) and 'mq' not in c.lower()]
    ntn_col = ntn_candidates[0]
    geo_cols = [c for c in [lista_code, 'Comune', 'Provincia', 'Regione'] if c in lista.columns]
    geo = lista[geo_cols].rename(columns={lista_code: 'municipality_code'}).copy()
    vals = res[[res_code, ntn_col]].rename(columns={res_code: 'municipality_code', ntn_col: 'ntn'}).copy()
    vals['ntn'] = pd.to_numeric(vals['ntn'], errors='coerce')
    if vals['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate NTN municipality key')
    tx = geo.merge(vals, on='municipality_code', how='left', validate='one_to_one')
    tx['year'] = year
    transaction_parts.append(tx)
transactions = pd.concat(transaction_parts, ignore_index=True)

population_parts = []
for folder in sorted(p for p in POPULATION_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year = int(folder.name)
    path = next(folder.glob('*_Comuni.csv'))
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', usecols=['Codice comune', 'Età', 'Totale'])
    df.columns = ['municipality_code', 'age', 'population']
    df['municipality_code'] = df['municipality_code'].astype('string').str.strip()
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['population'] = pd.to_numeric(df['population'], errors='coerce')
    totals = df.loc[df['age'].eq(999), ['municipality_code', 'population']].copy()
    totals['year'] = year
    population_parts.append(totals)
population = pd.concat(population_parts, ignore_index=True)

market = price_panel.merge(transactions[['year', 'municipality_code', 'ntn']], on=['year', 'municipality_code'], how='left', validate='many_to_one')
market = market.merge(population, on=['year', 'municipality_code'], how='left', validate='many_to_one')
market['period'] = market['year'].astype(str) + '-' + market['semester']
assert not market.duplicated(['year', 'semester', 'municipality_code']).any()
print(f'Market panel rows: {len(market):,}')
display(market.head())

## 3. Latest-period market KPIs

We distinguish **price level**, **transaction activity** and **market size**. A high price market is not necessarily a high-volume market.

In [ ]:
latest_period = market['period'].max()
latest = market[market['period'].eq(latest_period)].copy()
latest['price_percentile'] = latest['price_m2'].rank(pct=True) * 100
latest['population_percentile'] = latest['population'].rank(pct=True) * 100
latest['ntn_percentile'] = latest['ntn'].rank(pct=True) * 100

regional_kpi = (latest.groupby('region', dropna=False)
    .agg(median_price_m2=('price_m2', 'median'), total_ntn=('ntn', 'sum'), population=('population', 'sum'),
         municipalities=('municipality_code', 'nunique'), observations=('municipality_code', 'size'))
    .assign(ntn_per_1000_inhabitants=lambda x: x['total_ntn'] / x['population'] * 1000)
    .sort_values('median_price_m2', ascending=False))
display(regional_kpi.head(20))

## 4. Regional price and transaction rankings

Separate rankings avoid conflating expensive markets with liquid markets.

In [ ]:
print('Top regions by price level')
display(regional_kpi[['median_price_m2', 'municipalities']].head(10))

print('Top regions by transaction volume')
display(regional_kpi.sort_values('total_ntn', ascending=False)[['total_ntn', 'median_price_m2', 'municipalities']].head(10))

print('Top regions by transactions per 1,000 inhabitants')
display(regional_kpi.sort_values('ntn_per_1000_inhabitants', ascending=False)[['ntn_per_1000_inhabitants', 'total_ntn', 'population']].head(10))

## 5. Municipality ranking

Municipality rankings are restricted to observations with both price and population available. Very small municipalities should be interpreted carefully because percentage changes can be unstable.

In [ ]:
municipality_rank = latest.dropna(subset=['price_m2', 'population']).copy()
print('Most expensive municipalities')
display(municipality_rank.nlargest(20, 'price_m2')[['municipality_code', 'region', 'price_m2', 'population', 'ntn']])

print('Largest municipalities by population')
display(municipality_rank.nlargest(20, 'population')[['municipality_code', 'region', 'population', 'price_m2', 'ntn']])

print('Largest municipalities by NTN')
display(municipality_rank.nlargest(20, 'ntn')[['municipality_code', 'region', 'ntn', 'population', 'price_m2']])

## 6. Market-size segmentation

We classify municipalities into four descriptive segments using the median of price and NTN as sample-dependent cut-offs: **high-price/high-volume**, **high-price/low-volume**, **low-price/high-volume**, and **low-price/low-volume**.

In [ ]:
segment = municipality_rank.dropna(subset=['price_m2', 'ntn']).copy()
price_cut = segment['price_m2'].median()
ntn_cut = segment['ntn'].median()
segment['market_segment'] = np.select(
    [segment['price_m2'].ge(price_cut) & segment['ntn'].ge(ntn_cut),
     segment['price_m2'].ge(price_cut) & segment['ntn'].lt(ntn_cut),
     segment['price_m2'].lt(price_cut) & segment['ntn'].ge(ntn_cut)],
    ['High price / High volume', 'High price / Low volume', 'Low price / High volume'],
    default='Low price / Low volume')

display(segment['market_segment'].value_counts().rename_axis('segment').to_frame('municipalities'))
display(segment.groupby('market_segment').agg(municipalities=('municipality_code','size'), median_price_m2=('price_m2','median'), median_ntn=('ntn','median')).sort_values('median_price_m2', ascending=False))

fig, ax = plt.subplots(figsize=(10, 7))
for label, group in segment.groupby('market_segment'):
    ax.scatter(group['price_m2'], group['ntn'], label=label, alpha=0.45)
ax.axvline(price_cut, linestyle='--', linewidth=1)
ax.axhline(ntn_cut, linestyle='--', linewidth=1)
ax.set_title(f'Municipality Market Segmentation — {latest_period}')
ax.set_xlabel('Median OMI purchase quotation (€ / m²)')
ax.set_ylabel('NTN')
ax.legend()
fig.tight_layout()
plt.show()

## 7. Price and population dynamics

For each municipality we compare the latest available price change with the corresponding annual population change. The resulting quadrants are descriptive indicators of market-demographic alignment.

In [ ]:
market = market.sort_values(['municipality_code', 'period']).copy()
market['price_yoy_pct'] = market.groupby('municipality_code')['price_m2'].pct_change() * 100
market['population_yoy_pct'] = market.groupby('municipality_code')['population'].pct_change() * 100
dynamics = market[market['period'].eq(latest_period)].dropna(subset=['price_yoy_pct', 'population_yoy_pct']).copy()

price_growth_cut = dynamics['price_yoy_pct'].median()
population_growth_cut = dynamics['population_yoy_pct'].median()
dynamics['dynamic_segment'] = np.select(
    [dynamics['price_yoy_pct'].ge(price_growth_cut) & dynamics['population_yoy_pct'].ge(population_growth_cut),
     dynamics['price_yoy_pct'].ge(price_growth_cut) & dynamics['population_yoy_pct'].lt(population_growth_cut),
     dynamics['price_yoy_pct'].lt(price_growth_cut) & dynamics['population_yoy_pct'].ge(population_growth_cut)],
    ['Price growth / Population growth', 'Price growth / Population decline', 'Price decline / Population growth'],
    default='Price decline / Population decline')

display(dynamics['dynamic_segment'].value_counts().rename_axis('segment').to_frame('municipalities'))

fig, ax = plt.subplots(figsize=(10, 7))
for label, group in dynamics.groupby('dynamic_segment'):
    ax.scatter(group['population_yoy_pct'], group['price_yoy_pct'], label=label, alpha=0.45)
ax.axvline(population_growth_cut, linestyle='--', linewidth=1)
ax.axhline(price_growth_cut, linestyle='--', linewidth=1)
ax.set_title(f'Population vs Price Dynamics — {latest_period}')
ax.set_xlabel('Population YoY (%)')
ax.set_ylabel('OMI price YoY (%)')
ax.legend()
fig.tight_layout()
plt.show()

## 8. Market concentration

The share of national NTN generated by the largest municipalities provides a simple concentration indicator. It should not be interpreted as a formal concentration index unless the methodology is extended accordingly.

In [ ]:
national_ntn = segment['ntn'].sum()
largest = segment.sort_values('ntn', ascending=False).copy()
for n in [5, 10, 20, 50]:
    share = largest.head(n)['ntn'].sum() / national_ntn * 100 if national_ntn else np.nan
    print(f'Top {n} municipalities NTN share: {share:.2f}%')

largest['national_ntn_share_pct'] = largest['ntn'] / national_ntn * 100 if national_ntn else np.nan
display(largest.head(20)[['municipality_code', 'region', 'ntn', 'national_ntn_share_pct', 'price_m2']])

## 9. Analytical conclusions and hand-off

### Reusable outputs
- `regional_kpi`: regional price, NTN and demographic KPIs;
- `municipality_rank`: latest municipality-level ranking;
- `segment`: market-size segmentation;
- `dynamics`: price/population growth quadrants.

### Recommended next step
The next analytical layer should move from descriptive segmentation to **statistical modelling**: estimate price and transaction dynamics using lagged population, market size and regional controls, while explicitly handling missingness and municipality coverage changes.

### Methodological limits
OMI quotations are value ranges rather than observed transaction prices; NTN measures transaction volume rather than price; population is annual and is not a direct measure of housing demand. Municipality rankings can also be affected by coverage, structural changes and small denominators. Therefore, the notebook is suitable for exploratory market analysis, not causal inference.